# HistoWeave · 30 分钟 Workshop（一键路径）

**目标：** 安装验证 → 出 HTML 报告 → 对比域方法 → 了解成熟度与推荐。

| 时段 | 内容 |
|------|------|
| 0–5′ | 定位：编排 / 评测层，不是又一个 SOTA 算法 |
| 5–10′ | 安装检查 + Demo 流水线 |
| 10–18′ | `kmeans` vs `spectral` ARI |
| 18–25′ | `list-methods` 与 validated 证据 |
| 25–30′ | 推荐 API / 真实数据下一步 |

中文快速入门：`docs/zh/quickstart.md`  
包名：`histoweave-spatial` · Python ≥ 3.11

## 0. 安装（首次运行）

在终端执行一次即可：

```bash
pip install "histoweave-spatial[scanpy]"
```

若在本仓库源码上开发：

```bash
pip install -e ".[scanpy]"
```

In [ ]:
# 环境自检
import sys

print("Python", sys.version.split()[0])

import histoweave as ts

print("HistoWeave", getattr(ts, "__version__", "(dev)"))

from histoweave.plugins import list_methods

n = len(list_methods())
print(f"Registered methods: {n}")
assert n >= 10, "registry looks empty — reinstall histoweave-spatial[scanpy]"

## 1. 60 秒 Demo：合成数据 → 报告

默认流水线：QC → 归一化 → 域检测 → 注释，并写出自包含 HTML。

In [ ]:
from pathlib import Path

import histoweave as ts

data = ts.datasets.make_synthetic(n_cells=600, n_genes=40, n_domains=3, seed=0)
print(data)

result = ts.run_pipeline(data, verbose=True)

report_path = Path("workshop_report.html")
out = ts.build_report(result, report_path)
print("Report written:", Path(out).resolve())

# Provenance
steps = result.uns.get("run_manifest", {}).get("steps", [])
for s in steps:
    print(f"  {s.get('category'):18s}  {s.get('method'):16s}  v{s.get('version')}")

## 2. 对比两个域检测方法

在有种植 `domain_truth` 的合成数据上比较 ARI。真实分析常无 domain GT——那时应看多方法一致性，而不是用 Leiden 冒充 domain 真值。

In [ ]:
import histoweave as ts
from histoweave._math import adjusted_rand_index
from histoweave.plugins import MethodCategory, create_method

data = ts.datasets.make_synthetic(n_cells=600, n_genes=40, n_domains=3, seed=1)
norm = create_method(MethodCategory.NORMALIZATION, "log1p_cp10k").run(data)

rows = []
for name in ("kmeans", "spectral", "gaussian_mixture"):
    out = create_method(
        MethodCategory.DOMAIN_DETECTION,
        name,
        n_domains=3,
        random_state=0,
    ).run(norm.copy())
    ari = adjusted_rand_index(
        out.obs["domain_truth"].to_numpy(),
        out.obs["domain"].to_numpy(),
    )
    rows.append((name, float(ari), int(out.obs["domain"].nunique())))
    print(f"{name:18s}  ARI={ari:.3f}  n_domains={out.obs['domain'].nunique()}")

best = max(rows, key=lambda r: r[1])
print(f"\nBest on this toy draw: {best[0]} (ARI={best[1]:.3f})")
print("(Toy ARI is not a paper-grade ranking — see multi-dataset validation reports.)")

## 3. 方法目录与成熟度

生产用户优先看 **validated**；SOTA 外部包缺依赖时必须 **fail-closed**（明确报错，不静默换玩具实现）。

In [ ]:
from collections import Counter

from histoweave.plugins import list_methods

methods = list_methods()
by_mat = Counter(m.get("maturity", "?") for m in methods)
print("Maturity counts:", dict(sorted(by_mat.items())))

print("\nValidated methods (sample):")
validated = sorted(m["name"] for m in methods if m.get("maturity") == "validated")
print(", ".join(validated) if validated else "(none listed)")

print("\nDomain detection methods:")
for m in sorted(methods, key=lambda x: x["name"]):
    if m.get("category") == "domain_detection" or str(m.get("category", "")).endswith(
        "domain_detection"
    ):
        print(f"  {m['name']:20s}  {m.get('maturity', '?'):12s}  {m.get('summary', '')[:60]}")

## 4. 迷你基准（可选，~10–30s）

对当前合成表跑 domain_detection leaderboard。

In [ ]:
import histoweave as ts
from histoweave.benchmark import domain_detection_task, run_benchmark

data = ts.datasets.make_synthetic(n_cells=400, n_genes=30, n_domains=3, seed=2)
bench = run_benchmark(domain_detection_task(data))
print("Leaderboard (domain_detection on synthetic):")
for row in bench.leaderboard[:8]:
    print(f"  {row['rank']}. {row['method']:18s}  score={row['score']}")

## 5. 方法推荐（若有 knowledge base）

仓库内若存在 `figure3_results/landscape.json` 则运行推荐；否则跳过并提示如何生成。

In [ ]:
from pathlib import Path

import histoweave as ts

kb = None
here = Path.cwd()
for root in [here, *here.parents]:
    cand = root / "figure3_results" / "landscape.json"
    if cand.is_file():
        kb = cand
        break

if kb is None:
    print("No landscape.json found — skip recommend.")
    print("Generate one with: histoweave benchmark --task domain_detection --out landscape.json")
else:
    from histoweave.benchmark import AnalysisTask, MethodRecommender

    data = ts.datasets.make_synthetic(seed=0)
    rec = MethodRecommender(str(kb)).recommend(
        data,
        task=AnalysisTask.SPATIAL_DOMAIN,
        platform="visium",
        spatial_context_policy="high",
    )
    print("Knowledge base:", kb)
    print(rec.summary() if hasattr(rec, "summary") else rec)
    print("beats_global_best_baseline:", getattr(rec, "beats_global_best_baseline", "n/a"))
    print("warnings:", getattr(rec, "warnings", None))

## 6. 下一步（真实数据 / 贡献）

```bash
# Visium
histoweave ingest --input /path/to/spaceranger/outs --assay visium --out sample.ttab

# CLI demo
histoweave run --demo --out report.html

# 验证报告（仓库内）
# docs/methods/validation/index.md
# docs/zh/quickstart.md
```

**记住：** 缺官方 SOTA 包时应报错退出；推荐打不过 global-best 也是合法科学结论。

欢迎贡献插件：见仓库 `CONTRIBUTING.md`。

In [ ]:
print("Workshop complete. Open workshop_report.html if generated above.")
print("Docs: docs/zh/quickstart.md · docs/methods/validation/index.md")